# KCS Cases — OpenShift Troubleshooting Classifier

Fine-tune DistilBERT to classify OpenShift/Kubernetes issues into 6 categories based on real Red Hat KCS (Knowledge Center Solutions) patterns.

### Labels

| ID | Label | Description |
|----|-------|-------------|
| 0 | `crashloop_backoff` | Pod stuck in CrashLoopBackOff restart loop |
| 1 | `image_pull_error` | Container image cannot be pulled |
| 2 | `route_503` | OpenShift route returning 503 Service Unavailable |
| 3 | `pvc_pending` | PersistentVolumeClaim stuck in Pending |
| 4 | `quota_exceeded` | Resource quota preventing pod scheduling |
| 5 | `scheduling_constraint` | Pod unschedulable due to taints/affinity/resources |

### Dataset

- 300 examples (50 per class)
- Each example includes structured fields: Deployment, Pod state, Events, Logs
- Solutions sourced from Red Hat KCS articles

---
## 0. Install Dependencies

In [80]:
!pip install -q transformers datasets evaluate accelerate boto3 s3fs

9064.35s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


---
## 1. Imports and GPU Check

In [81]:
import os
import json
import numpy as np
import torch
import evaluate
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

---
## 2. Configuration

In [82]:
MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = 6
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
SEED = 42
OUTPUT_DIR = "./results-kcs"
SAVE_DIR = "./kcs-classifier"
DATA_FILE = "data/ocp-issues-v2.csv"

---
## 3. Define and Save Label Mapping

The model needs integer labels. We define a bidirectional mapping and save it to `labels.json` so it can be used at inference time.

In [83]:
label2id = {
    "crashloop_backoff": 0,
    "image_pull_error": 1,
    "route_503": 2,
    "pvc_pending": 3,
    "quota_exceeded": 4,
    "scheduling_constraint": 5,
}

id2label = {v: k for k, v in label2id.items()}

labels_path = Path("data/labels.json")
labels_path.parent.mkdir(parents=True, exist_ok=True)
with open(labels_path, "w") as f:
    json.dump({"label2id": label2id, "id2label": {str(k): v for k, v in id2label.items()}}, f, indent=2)

print(f"Label mapping saved to {labels_path}")
print(f"Labels: {label2id}")

Label mapping saved to data/labels.json
Labels: {'crashloop_backoff': 0, 'image_pull_error': 1, 'route_503': 2, 'pvc_pending': 3, 'quota_exceeded': 4, 'scheduling_constraint': 5}


---
## 4. Load the Dataset

Load the KCS cases CSV and split into 80% train / 20% test.

In [86]:
dataset = load_dataset("csv", data_files=DATA_FILE)["train"]
dataset = dataset.train_test_split(test_size=0.2, seed=SEED)

train_ds = dataset["train"]
eval_ds = dataset["test"]

print(f"Training samples:   {len(train_ds)}")
print(f"Evaluation samples: {len(eval_ds)}")
print(f"Columns: {train_ds.column_names}")
print(f"\nSample:")
print(f"  Text:  {train_ds[0]['text'][:120]}...")
print(f"  Label: {train_ds[0]['label']}")

Generating train split: 0 examples [00:00, ? examples/s]

Training samples:   240
Evaluation samples: 60
Columns: ['text', 'label', 'solution']

Sample:
  Text:  Deployment: logstash; Pod state: ImagePullBackOff; Events: Failed to pull image docker.elastic.co/logstash/logstash:8.99...
  Label: image_pull_error


---
## 5. Convert String Labels to Integers

The model expects numeric labels. Map each string label to its integer ID.

In [87]:
def encode_labels(example):
    example["label"] = label2id[example["label"]]
    return example

train_ds = train_ds.map(encode_labels)
eval_ds = eval_ds.map(encode_labels)

from collections import Counter
train_counts = Counter(train_ds["label"])
print("Train label distribution:")
for label_id in sorted(train_counts):
    print(f"  {id2label[label_id]}: {train_counts[label_id]}")

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Train label distribution:
  crashloop_backoff: 42
  image_pull_error: 40
  route_503: 37
  pvc_pending: 41
  quota_exceeded: 37
  scheduling_constraint: 43


---
## 6. Tokenization

In [88]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

train_ds = train_ds.map(tokenize_function, batched=True, desc="Tokenizing train")
eval_ds = eval_ds.map(tokenize_function, batched=True, desc="Tokenizing eval")

print(f"Tokenized fields: {train_ds.column_names}")
print(f"Token count for first sample: {len(train_ds[0]['input_ids'])}")

Tokenizing train:   0%|          | 0/240 [00:00<?, ? examples/s]

Tokenizing eval:   0%|          | 0/60 [00:00<?, ? examples/s]

Tokenized fields: ['text', 'label', 'solution', 'input_ids', 'token_type_ids', 'attention_mask']
Token count for first sample: 46


---
## 7. Load the Pretrained Model

Load DistilBERT with a 6-class classification head. We pass `label2id` and `id2label` so the model config stores the label names.

In [89]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    label2id=label2id,
    id2label=id2label,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters:     66,958,086
Trainable parameters: 66,958,086


---
## 8. Define Evaluation Metrics

For multi-class classification we track both accuracy and per-class F1.

In [92]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    return acc

---
## 9. Configure Training

| Parameter | Value | Why |
|-----------|-------|-----|
| `num_train_epochs` | 10 | Small dataset needs more passes |
| `per_device_train_batch_size` | 16 | Fits in GPU memory for DistilBERT |
| `learning_rate` | 2e-5 | Standard for BERT fine-tuning |
| `weight_decay` | 0.01 | Regularization to prevent overfitting |

In [93]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none",
    seed=SEED,
)

---
## 10. Train

In [94]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print("Starting fine-tuning on KCS cases...")
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"  Training loss:    {train_result.metrics['train_loss']:.4f}")
print(f"  Training runtime: {train_result.metrics['train_runtime']:.1f}s")
print(f"  Samples/second:   {train_result.metrics['train_samples_per_second']:.1f}")

Starting fine-tuning on KCS cases...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.779409,1.595666,0.750000
2,1.312947,1.098945,0.983333
3,1.015751,0.659594,0.983333
4,0.531336,0.409518,0.983333
5,0.393640,0.280871,0.983333
6,0.243767,0.212762,0.983333
7,0.198529,0.178693,0.983333
8,0.136118,0.159248,0.983333
9,0.126077,0.151849,0.983333
10,0.132454,0.149133,0.983333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete!
  Training loss:    0.5879
  Training runtime: 26.1s
  Samples/second:   91.9


---
## 11. Evaluation Results

In [95]:
eval_metrics = [m for m in trainer.state.log_history if "eval_accuracy" in m]

if eval_metrics:
    print("Evaluation history:")
    for m in eval_metrics:
        print(f"  Epoch {m.get('epoch', '?')}: accuracy={m['eval_accuracy']:.4f}  loss={m['eval_loss']:.4f}")
    print(f"\nBest accuracy: {max(m['eval_accuracy'] for m in eval_metrics):.4f}")
else:
    print("No evaluation metrics found.")

Evaluation history:
  Epoch 1.0: accuracy=0.7500  loss=1.5957
  Epoch 2.0: accuracy=0.9833  loss=1.0989
  Epoch 3.0: accuracy=0.9833  loss=0.6596
  Epoch 4.0: accuracy=0.9833  loss=0.4095
  Epoch 5.0: accuracy=0.9833  loss=0.2809
  Epoch 6.0: accuracy=0.9833  loss=0.2128
  Epoch 7.0: accuracy=0.9833  loss=0.1787
  Epoch 8.0: accuracy=0.9833  loss=0.1592
  Epoch 9.0: accuracy=0.9833  loss=0.1518
  Epoch 10.0: accuracy=0.9833  loss=0.1491

Best accuracy: 0.9833


---
## 12. Save the Model

In [96]:
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import shutil
shutil.copy("data/labels.json", Path(SAVE_DIR) / "labels.json")

saved_files = os.listdir(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}/")
print(f"Files: {saved_files}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./kcs-classifier/
Files: ['training_args.bin', 'labels.json', 'model.safetensors', 'tokenizer_config.json', 'config.json', 'tokenizer.json']


---
## 13. Test Inference

Run the classifier on sample OpenShift issue descriptions.

In [97]:
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
model.eval()

test_cases = [
    "Deployment: api-server; Pod state: CrashLoopBackOff; Restarts: 15; Events: Back-off restarting failed container; Logs: FATAL: password authentication failed for user postgres",
    "Deployment: web-frontend; Pod state: ImagePullBackOff; Events: Failed to pull image registry.internal/frontend:v3; unauthorized: authentication required",
    "Route: payment-route; HTTP 503; Service endpoints: 0; Pods ready: 0/2; Deployment replicas: 2 desired 0 available",
    "PVC: data-volume; Status: Pending; Events: storageclass fast-ssd not found; No default StorageClass configured",
    "Deployment: ml-worker; Pod state: Pending; Events: exceeded quota: gpu-quota; requested: nvidia.com/gpu=2; used: 4; limited: 4",
    "Deployment: cache-server; Pod state: Pending; Events: 0/6 nodes are available: 3 node(s) had taint dedicated=gpu:NoSchedule that the pod didn't tolerate; 3 had master taint",
]

print("Inference results:\n")
for text in test_cases:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)[0]
    predicted_id = torch.argmax(probs).item()
    predicted_label = id2label[predicted_id]
    confidence = probs[predicted_id].item()
    print(f"  [{predicted_label} ({confidence:.1%})] {text[:80]}...")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Inference results:

  [crashloop_backoff (92.5%)] Deployment: api-server; Pod state: CrashLoopBackOff; Restarts: 15; Events: Back-...
  [image_pull_error (93.1%)] Deployment: web-frontend; Pod state: ImagePullBackOff; Events: Failed to pull im...
  [route_503 (90.7%)] Route: payment-route; HTTP 503; Service endpoints: 0; Pods ready: 0/2; Deploymen...
  [pvc_pending (92.1%)] PVC: data-volume; Status: Pending; Events: storageclass fast-ssd not found; No d...
  [quota_exceeded (92.0%)] Deployment: ml-worker; Pod state: Pending; Events: exceeded quota: gpu-quota; re...
  [scheduling_constraint (90.4%)] Deployment: cache-server; Pod state: Pending; Events: 0/6 nodes are available: 3...


---
## 14. Upload to MinIO

In [98]:
import boto3
from datetime import datetime, timezone
from botocore.client import Config

BUCKET_NAME = os.environ.get("AWS_S3_BUCKET", "fine-tuning")
S3_PREFIX = "kcs-classifier/v1"


def build_s3_client():
    return boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        config=Config(signature_version="s3v4"),
        region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    )


def write_metadata(target_dir: Path, export_type: str, base_model: str):
    target_dir.mkdir(parents=True, exist_ok=True)
    metadata = {
        "export_type": export_type,
        "base_model": base_model,
        "num_labels": NUM_LABELS,
        "labels": label2id,
        "dataset": DATA_FILE,
        "epochs": EPOCHS,
        "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    with open(target_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)


def upload_directory_to_s3(local_dir: Path, bucket: str, prefix: str):
    s3 = build_s3_client()
    for file_path in local_dir.rglob("*"):
        if file_path.is_file():
            s3_key = f"{prefix}/{file_path.relative_to(local_dir).as_posix()}"
            print(f"Uploading {file_path.name} -> s3://{bucket}/{s3_key}")
            s3.upload_file(str(file_path), bucket, s3_key)
    print("Upload finished.")


def list_s3_prefix(bucket: str, prefix: str):
    s3 = build_s3_client()
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    print(f"\nObjects in s3://{bucket}/{prefix}")
    for obj in response.get("Contents", []):
        print(f"- {obj['Key']} ({obj['Size']} bytes)")


save_dir = Path(SAVE_DIR)
write_metadata(save_dir, "huggingface-sequence-classification", MODEL_NAME)
upload_directory_to_s3(save_dir, BUCKET_NAME, S3_PREFIX)
list_s3_prefix(BUCKET_NAME, S3_PREFIX)

Uploading training_args.bin -> s3://fine-tuning/kcs-classifier/v1/training_args.bin
Uploading labels.json -> s3://fine-tuning/kcs-classifier/v1/labels.json
Uploading metadata.json -> s3://fine-tuning/kcs-classifier/v1/metadata.json
Uploading model.safetensors -> s3://fine-tuning/kcs-classifier/v1/model.safetensors
Uploading tokenizer_config.json -> s3://fine-tuning/kcs-classifier/v1/tokenizer_config.json
Uploading config.json -> s3://fine-tuning/kcs-classifier/v1/config.json
Uploading tokenizer.json -> s3://fine-tuning/kcs-classifier/v1/tokenizer.json
Upload finished.

Objects in s3://fine-tuning/kcs-classifier/v1
- kcs-classifier/v1/config.json (1023 bytes)
- kcs-classifier/v1/labels.json (362 bytes)
- kcs-classifier/v1/metadata.json (404 bytes)
- kcs-classifier/v1/model.safetensors (267844872 bytes)
- kcs-classifier/v1/tokenizer.json (711494 bytes)
- kcs-classifier/v1/tokenizer_config.json (322 bytes)
- kcs-classifier/v1/training_args.bin (5201 bytes)


---
## Next Steps

1. **More data** — Add more examples per class (aim for 200+ per class)
2. **Data augmentation** — Paraphrase existing examples with different wording
3. **Larger model** — Try `bert-base-uncased` or `roberta-base` for better accuracy
4. **Solution generation** — Fine-tune a generative LLM to output the `solution` field given the `text`
5. **Deploy** — Serve the model via KServe on OpenShift AI for real-time classification